[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/06_tag_2_3_aktivierungsfunktionen.ipynb)

# Tag 2/3 - 06 Aktivierungsfunktionen

Aktivierungsfunktionen machen ein neuronales Netz nichtlinear. In diesem Notebook bleibt der Datensatz gleich und wir ändern bewusst nur die Aktivierung in den Hidden Layern.

Zusätzlich zeichnen wir am Ende die **Trennbereiche** der Modelle. So sieht man nicht nur die Zahl `accuracy`, sondern auch die gelernte Entscheidungsfläche.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
tf.keras.utils.set_random_seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


## Formen typischer Aktivierungsfunktionen

Die Aktivierungsfunktion bekommt die lineare Voraktivierung `z` und gibt die Aktivierung an den nächsten Layer weiter.


In [ ]:
z = np.linspace(-5, 5, 400)
plt.plot(z, 1 / (1 + np.exp(-z)), label="sigmoid")
plt.plot(z, np.tanh(z), label="tanh")
plt.plot(z, np.maximum(0, z), label="relu")
plt.plot(z, np.where(z >= 0, z, 0.1 * z), label="leaky_relu")
plt.axhline(0, color="black", linewidth=0.8)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Formen typischer Aktivierungsfunktionen")
plt.xlabel("z")
plt.ylabel("f(z)")
plt.legend()
plt.show()


## Datensatz und Hilfsfunktionen

`make_moons` ist nicht linear trennbar. Ein einzelnes Neuron reicht dafür nicht, ein kleines MLP aber schon. Alle Modelle haben dieselbe Struktur: zwei Hidden Layer und ein Sigmoid-Output.


In [ ]:
X, y = make_moons(n_samples=900, noise=0.25, random_state=RANDOM_STATE)
X = StandardScaler().fit_transform(X).astype("float32")
y = y.astype("float32")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=18, edgecolor="black", alpha=0.75)
plt.title("Two-Moons-Datensatz")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

trained_models = {}
results = {}


def train_and_report(model, name, epochs=45):
    model.compile(optimizer=keras.optimizers.Adam(0.02), loss="binary_crossentropy", metrics=["accuracy"])
    history = model.fit(X_train, y_train, validation_split=0.25, epochs=epochs, batch_size=32, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    trained_models[name] = model
    results[name] = {"history": history, "test_accuracy": test_accuracy}
    print(f"{name:>12}: Test Loss={test_loss:.3f}, Test Accuracy={test_accuracy:.1%}")
    return history


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="validation")
    axes[0].set_title(f"{title}: Loss")
    axes[0].set_xlabel("Epoche")
    axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="validation")
    axes[1].set_title(f"{title}: Accuracy")
    axes[1].set_xlabel("Epoche")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


def plot_boundary(model, title, ax):
    xx, yy = np.meshgrid(
        np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 170),
        np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 170),
    )
    grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
    proba = model.predict(grid, verbose=0).reshape(xx.shape)
    ax.contourf(xx, yy, proba, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.35)
    ax.contour(xx, yy, proba, levels=[0.5], colors="black", linewidths=1.5)
    ax.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap="coolwarm", s=14, edgecolor="black")
    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


## ReLU

ReLU ist eine sehr häufige Standardwahl. Negative Werte werden zu 0, positive Werte bleiben erhalten.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
relu_model = keras.Sequential(
    [
        keras.Input(shape=(2,), name="features"),
        # Parameter im Fokus: activation="relu".
        layers.Dense(16, activation="relu", name="hidden_relu_1"),
        layers.Dense(16, activation="relu", name="hidden_relu_2"),
        layers.Dense(1, activation="sigmoid", name="output"),
    ]
)
relu_history = train_and_report(relu_model, "relu")
plot_history(relu_history, "ReLU")


## tanh

`tanh` gibt Werte zwischen -1 und 1 aus. Das kann stabil sein, ist aber in MLPs oft langsamer als ReLU.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
tanh_model = keras.Sequential(
    [
        keras.Input(shape=(2,), name="features"),
        # Parameter im Fokus: activation="tanh".
        layers.Dense(16, activation="tanh", name="hidden_tanh_1"),
        layers.Dense(16, activation="tanh", name="hidden_tanh_2"),
        layers.Dense(1, activation="sigmoid", name="output"),
    ]
)
tanh_history = train_and_report(tanh_model, "tanh")
plot_history(tanh_history, "tanh")


## Sigmoid im Hidden Layer

Sigmoid ist im Output-Layer für binäre Klassifikation sinnvoll. In Hidden Layern kann Sigmoid schneller sättigen: Dann werden Gradienten klein.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
sigmoid_model = keras.Sequential(
    [
        keras.Input(shape=(2,), name="features"),
        # Parameter im Fokus: activation="sigmoid" im Hidden Layer.
        layers.Dense(16, activation="sigmoid", name="hidden_sigmoid_1"),
        layers.Dense(16, activation="sigmoid", name="hidden_sigmoid_2"),
        layers.Dense(1, activation="sigmoid", name="output"),
    ]
)
sigmoid_history = train_and_report(sigmoid_model, "sigmoid")
plot_history(sigmoid_history, "Sigmoid")


## Leaky ReLU

Leaky ReLU lässt links von 0 eine kleine Steigung übrig. Dadurch bekommen auch negative Voraktivierungen noch einen Gradienten.


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_STATE)
leaky_model = keras.Sequential(
    [
        keras.Input(shape=(2,), name="features"),
        layers.Dense(16, name="hidden_leaky_1_linear"),
        # Parameter im Fokus: negative_slope ist die Steigung links von 0.
        layers.LeakyReLU(negative_slope=0.1, name="hidden_leaky_1"),
        layers.Dense(16, name="hidden_leaky_2_linear"),
        layers.LeakyReLU(negative_slope=0.1, name="hidden_leaky_2"),
        layers.Dense(1, activation="sigmoid", name="output"),
    ]
)
leaky_history = train_and_report(leaky_model, "leaky_relu")
plot_history(leaky_history, "Leaky ReLU")


## Custom Activation: Swish

Eigene Aktivierungsfunktionen können in Keras sehr kurz geschrieben werden. Wichtig ist aber: **Die Funktion bekommt TensorFlow-Tensoren und muss mit TensorFlow-Operationen arbeiten.**

Gut geeignet sind zum Beispiel:

- `tf.nn.sigmoid(x)`
- `tf.nn.relu(x)`
- `tf.maximum(...)`
- `tf.where(...)`
- normale Tensor-Operationen wie `x * ...` oder `x + ...`

Nicht sinnvoll sind hier NumPy-Operationen, `math.exp`, Python-Listenlogik oder Schleifen, die einzelne Werte aus dem Tensor herausziehen. Dann kann TensorFlow den Rechengraphen und die Gradienten für Backpropagation nicht zuverlässig bilden.

Das folgende Beispiel funktioniert, weil `tf.nn.sigmoid` eine TensorFlow-Operation ist und TensorFlow davon automatisch die Ableitung kennt:

`swish(x) = x * sigmoid(x)`


In [ ]:
def swish(x):
    # Custom Activation im Fokus:
    # x ist ein TensorFlow-Tensor, deshalb nutzen wir TensorFlow-Operationen.
    return x * tf.nn.sigmoid(x)


tf.keras.utils.set_random_seed(RANDOM_STATE)
custom_model = keras.Sequential(
    [
        keras.Input(shape=(2,), name="features"),
        layers.Dense(16, activation=swish, name="hidden_swish_1"),
        layers.Dense(16, activation=swish, name="hidden_swish_2"),
        layers.Dense(1, activation="sigmoid", name="output"),
    ]
)
custom_history = compile_and_train(custom_model)
results["custom_swish"] = {"model": custom_model, "history": custom_history}
plot_history(custom_history, "Custom Swish")


## Vergleich: Validation Accuracy und Trennbereiche

Die Entscheidungsflächen zeigen, wie unterschiedlich die Modelle denselben Datensatz aufteilen.


In [ ]:
plt.figure(figsize=(8, 4))
for name, result in results.items():
    plt.plot(result["history"].history["val_accuracy"], label=name)
plt.title("Validation Accuracy im Vergleich")
plt.xlabel("Epoche")
plt.ylabel("Validation Accuracy")
plt.ylim(0.5, 1.02)
plt.legend()
plt.show()

names = list(trained_models.keys())
fig, axes = plt.subplots(1, len(names), figsize=(4 * len(names), 3.8), sharex=True, sharey=True)
for ax, name in zip(axes, names):
    plot_boundary(trained_models[name], name, ax)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(results.keys(), [result["test_accuracy"] for result in results.values()])
plt.ylim(0.5, 1.0)
plt.ylabel("Test Accuracy")
plt.title("Aktivierungsfunktionen im Testvergleich")
plt.xticks(rotation=20)
plt.show()
